# Route Optimization

Original script by haythamomar — source: https://github.com/gkhayes/mlrose/blob/master/tutorial_examples.ipynb

In [2]:
import six
import sys
sys.modules['sklearn.externals.six'] = six
import mlrose

## Travelling Salesman Problem (mlrose)

In [6]:
### travelling sales man problem 


# Create list of distances between pairs of cities
dist_list = [(0, 1, 3.1623), (0, 2, 4.1231), (0, 3, 5.8310), (0, 4, 4.2426), (0, 5, 5.3852), 
             (0, 6, 4.0000), (0, 7, 2.2361), (1, 2, 1.0000), (1, 3, 2.8284), (1, 4, 2.0000), 
             (1, 5, 4.1231), (1, 6, 4.2426), (1, 7, 2.2361), (2, 3, 2.2361), (2, 4, 2.2361),
             (2, 5, 4.4721), (2, 6, 5.0000), (2, 7, 3.1623), (3, 4, 2.0000), (3, 5, 3.6056), 
             (3, 6, 5.0990), (3, 7, 4.1231), (4, 5, 2.2361), (4, 6, 3.1623), (4, 7, 2.2361), 
             (5, 6, 2.2361), (5, 7, 3.1623), (6, 7, 2.2361)]

fit_function= mlrose.TravellingSales(distances= dist_list)

fit= mlrose.TSPOpt(length=8,fitness_fn=fit_function,maximize= False)

print(fit_function)
model= mlrose.genetic_alg(fit,mutation_prob=0.1,max_attempts=100,random_state=2)

KeyboardInterrupt: 

## OR-Tools: Vehicle Routing Problem with 4 Vehicles

Source: https://developers.google.com/optimization/routing/vrp

In [4]:
####OR tools

##Source:https://developers.google.com/optimization/routing/vrp

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

###Vehicle routing problem with 4 Vehicles
data = {}
data['distance_matrix'] = [
        [
            0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354,
            468, 776, 662
        ],
        [
            548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674,
            1016, 868, 1210
        ],
        [
            776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164,
            1130, 788, 1552, 754
        ],
        [
            696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822,
            1164, 560, 1358
        ],
        [
            582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708,
            1050, 674, 1244
        ],
        [
            274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628,
            514, 1050, 708
        ],
        [
            502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856,
            514, 1278, 480
        ],
        [
            194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320,
            662, 742, 856
        ],
        [
            308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662,
            320, 1084, 514
        ],
        [
            194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388,
            274, 810, 468
        ],
        [
            536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764,
            730, 388, 1152, 354
        ],
        [
            502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114,
            308, 650, 274, 844
        ],
        [
            388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194,
            536, 388, 730
        ],
        [
            354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0,
            342, 422, 536
        ],
        [
            468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536,
            342, 0, 764, 194
        ],
        [
            776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274,
            388, 422, 764, 0, 798
        ],
        [
            662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730,
            536, 194, 798, 0
        ],
    ]
data['num_vehicles'] = 4
data['depot'] = 0


manager= pywrapcp.RoutingIndexManager(len(data['distance_matrix']),data['num_vehicles'],data['depot'])

## make routing model

routing= pywrapcp.RoutingModel(manager)


def distance_callback(from_index, to_index):
        """Returns the distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['distance_matrix'][from_node][to_node]


transit_callback_index = routing.RegisterTransitCallback(distance_callback)

routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


###setting constraints


    # Add Distance constraint.
dimension_name = 'Distance'
routing.AddDimension(
        transit_callback_index,
        0,  # no slack
        3000,  # vehicle maximum travel distance
        True,  # start cumul to zero
        dimension_name)
distance_dimension = routing.GetDimensionOrDie(dimension_name)
distance_dimension.SetGlobalSpanCostCoefficient(100)


    # Setting first solution heuristic.
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    # Solve the problem.
solution = routing.SolveWithParameters(search_parameters)

def get_routes(solution,routing,manager):
    routes=[]
    for node in range(routing.vehicles()):
        index= routing.Start(node)
        route= [manager.IndexToNode(index)]
        while not routing.IsEnd(index):
            index= solution.Value(routing.NextVar(index))
            route.append(manager.IndexToNode(index))
        routes.append(route)
    return(routes)
get_routes(solution,routing,manager)            

[[0, 9, 10, 2, 6, 5, 0],
 [0, 16, 14, 8, 0],
 [0, 7, 1, 4, 3, 0],
 [0, 13, 15, 11, 12, 0]]

## VRP with Capacity Constraints

In [ ]:
#### with capacity constraints

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

###4 vheiCLES
data = {}
data['distance_matrix'] = [
        [
            0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536, 502, 388, 354,
            468, 776, 662
        ],
        [
            548, 0, 684, 308, 194, 502, 730, 354, 696, 742, 1084, 594, 480, 674,
            1016, 868, 1210
        ],
        [
            776, 684, 0, 992, 878, 502, 274, 810, 468, 742, 400, 1278, 1164,
            1130, 788, 1552, 754
        ],
        [
            696, 308, 992, 0, 114, 650, 878, 502, 844, 890, 1232, 514, 628, 822,
            1164, 560, 1358
        ],
        [
            582, 194, 878, 114, 0, 536, 764, 388, 730, 776, 1118, 400, 514, 708,
            1050, 674, 1244
        ],
        [
            274, 502, 502, 650, 536, 0, 228, 308, 194, 240, 582, 776, 662, 628,
            514, 1050, 708
        ],
        [
            502, 730, 274, 878, 764, 228, 0, 536, 194, 468, 354, 1004, 890, 856,
            514, 1278, 480
        ],
        [
            194, 354, 810, 502, 388, 308, 536, 0, 342, 388, 730, 468, 354, 320,
            662, 742, 856
        ],
        [
            308, 696, 468, 844, 730, 194, 194, 342, 0, 274, 388, 810, 696, 662,
            320, 1084, 514
        ],
        [
            194, 742, 742, 890, 776, 240, 468, 388, 274, 0, 342, 536, 422, 388,
            274, 810, 468
        ],
        [
            536, 1084, 400, 1232, 1118, 582, 354, 730, 388, 342, 0, 878, 764,
            730, 388, 1152, 354
        ],
        [
            502, 594, 1278, 514, 400, 776, 1004, 468, 810, 536, 878, 0, 114,
            308, 650, 274, 844
        ],
        [
            388, 480, 1164, 628, 514, 662, 890, 354, 696, 422, 764, 114, 0, 194,
            536, 388, 730
        ],
        [
            354, 674, 1130, 822, 708, 628, 856, 320, 662, 388, 730, 308, 194, 0,
            342, 422, 536
        ],
        [
            468, 1016, 788, 1164, 1050, 514, 514, 662, 320, 274, 388, 650, 536,
            342, 0, 764, 194
        ],
        [
            776, 868, 1552, 560, 674, 1050, 1278, 742, 1084, 810, 1152, 274,
            388, 422, 764, 0, 798
        ],
        [
            662, 1210, 754, 1358, 1244, 708, 480, 856, 514, 468, 354, 844, 730,
            536, 194, 798, 0
        ],
    ]
data['num_vehicles'] = 4
data['depot'] = 0
data['demand']= [0,1,1,2,4,2,4,8,8,1,2,1,2,4,4,8,8]
data['Capacity']=[15,15,15,15]



manager= pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                      data['num_vehicles'],data['depot'])

## make routing model

routing= pywrapcp.RoutingModel(manager)


def distance_callback(from_index, to_index):
        """Returns the distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['distance_matrix'][from_node][to_node]
    
def demand_callback(from_index):
        """Returns the distance between the two nodes."""
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        return data['demand'][from_node]
    

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
demand_callback_index= routing.RegisterUnaryTransitCallback(demand_callback)

routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


###setting constraints


    # Add Distance constraint.
dimension_name = 'Distance'
routing.AddDimension(
        transit_callback_index,
        0,  # no slack
        3000,  # vehicle maximum travel distance
        True,  # start cumul to zero
        dimension_name)
distance_dimension = routing.GetDimensionOrDie(dimension_name)
distance_dimension.SetGlobalSpanCostCoefficient(100)

routing.AddDimensionWithVehicleCapacity(demand_callback_index,
                                        0, 
                                       data['Capacity'],
                                        True,
                                       'Capacity')


    # Setting first solution heuristic.
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    # Solve the problem.
solution = routing.SolveWithParameters(search_parameters)



def get_routes(solution,routing,manager):
    route_loads= list(dict())
    for v in range(routing.vehicles()):
        index= routing.Start(v)
        route= manager.IndexToNode(index)
        route_load= {}
        route_load[route]= data['demand'][route]
        while not routing.IsEnd(index):
            index= solution.Value(routing.NextVar(index))
            route_next= manager.IndexToNode(index)
            route_load[route_next]=data['demand'][route_next]
        route_loads.append(route_load)
    return(route_loads)
        
get_routes(solution, routing, manager)

### Add Distance per Route

In [ ]:
###with distance

def get_routes(solution,routing,manager):
    route_loads= list(dict())
    for v in range(routing.vehicles()):
        distance=0
        total_load= 0
        index= routing.Start(v)
        route= manager.IndexToNode(index)
        route_load= {}
        route_load[route]= data['demand'][route]
        while not routing.IsEnd(index):
            previous_index= index
            index= solution.Value(routing.NextVar(index))
            route_next= manager.IndexToNode(index)
            distance += routing.GetArcCostForVehicle(previous_index,index,v)
            route_load['Vehicle_distance']= 'Total distance for vehicle no {} is {}'.format(v,distance)
            total_load+= data['demand'][route_next]
            route_load[route_next]=data['demand'][route_next]
            route_load['total_load']= total_load
        route_loads.append(route_load)
    return(route_loads)
        
get_routes(solution, routing, manager)

## VRP with Time Windows

In [ ]:
##### with time windows 

data = {}
data['time_matrix'] = [
        [0, 6, 9, 8, 7, 3, 6, 2, 3, 2, 6, 6, 4, 4, 5, 9, 7],
        [6, 0, 8, 3, 2, 6, 8, 4, 8, 8, 13, 7, 5, 8, 12, 10, 14],
        [9, 8, 0, 11, 10, 6, 3, 9, 5, 8, 4, 15, 14, 13, 9, 18, 9],
        [8, 3, 11, 0, 1, 7, 10, 6, 10, 10, 14, 6, 7, 9, 14, 6, 16],
        [7, 2, 10, 1, 0, 6, 9, 4, 8, 9, 13, 4, 6, 8, 12, 8, 14],
        [3, 6, 6, 7, 6, 0, 2, 3, 2, 2, 7, 9, 7, 7, 6, 12, 8],
        [6, 8, 3, 10, 9, 2, 0, 6, 2, 5, 4, 12, 10, 10, 6, 15, 5],
        [2, 4, 9, 6, 4, 3, 6, 0, 4, 4, 8, 5, 4, 3, 7, 8, 10],
        [3, 8, 5, 10, 8, 2, 2, 4, 0, 3, 4, 9, 8, 7, 3, 13, 6],
        [2, 8, 8, 10, 9, 2, 5, 4, 3, 0, 4, 6, 5, 4, 3, 9, 5],
        [6, 13, 4, 14, 13, 7, 4, 8, 4, 4, 0, 10, 9, 8, 4, 13, 4],
        [6, 7, 15, 6, 4, 9, 12, 5, 9, 6, 10, 0, 1, 3, 7, 3, 10],
        [4, 5, 14, 7, 6, 7, 10, 4, 8, 5, 9, 1, 0, 2, 6, 4, 8],
        [4, 8, 13, 9, 8, 7, 10, 3, 7, 4, 8, 3, 2, 0, 4, 5, 6],
        [5, 12, 9, 14, 12, 6, 6, 7, 3, 3, 4, 7, 6, 4, 0, 9, 2],
        [9, 10, 18, 6, 8, 12, 15, 8, 13, 9, 13, 3, 4, 5, 9, 0, 9],
        [7, 14, 9, 16, 14, 8, 5, 10, 6, 5, 4, 10, 8, 6, 2, 9, 0],
       ]
data['time_windows'] = [
        (0, 5),  # depot
        (7, 12),  # 1
        (10, 15),  # 2
        (16, 18),  # 3
        (10, 13),  # 4
        (0, 5),  # 5
        (5, 10),  # 6
        (0, 4),  # 7
        (5, 10),  # 8
        (0, 3),  # 9
        (10, 16),  # 10
        (10, 15),  # 11
        (0, 5),  # 12
        (5, 10),  # 13
        (7, 8),  # 14
        (10, 15),  # 15
        (11, 15),  # 16
    ]
data['num_vehicles'] = 4
data['depot'] = 0


manager= pywrapcp.RoutingIndexManager(len(data['time_matrix']),
                                      data['num_vehicles'],data['depot'])


routing= pywrapcp.RoutingModel(manager)


def time_callback(from_index,to_index):
    from_node= manager.IndexToNode(from_index)
    to_node= manager.IndexToNode(to_index)
    return data['time_matrix'][from_node][to_node]

transit_callback_index= routing.RegisterTransitCallback(time_callback)

routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

time='Time'
routing.AddDimension( transit_callback_index
    , 30
    , 30, False, time)


time_dimension= routing.GetDimensionOrDie(time)

for node, time_window in enumerate(data['time_windows']):
    if node ==0:
        continue
    index= manager.NodeToIndex(node)
    time_dimension.CumulVar(index).SetRange(time_window[0],time_window[1])

for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        time_dimension.CumulVar(index).SetRange(data['time_windows'][0][0],
                                                data['time_windows'][0][1])

  
for i in range(data['num_vehicles']):
        routing.AddVariableMinimizedByFinalizer(
            time_dimension.CumulVar(routing.Start(i)))
        routing.AddVariableMinimizedByFinalizer(
            time_dimension.CumulVar(routing.End(i)))


   # Setting first solution heuristic.
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    # Solve the problem.
solution = routing.SolveWithParameters(search_parameters)

      
def get_routes(solution,routing,manager,dimension):
    all_list=[]
    for vehicle in range(routing.vehicles()):
        route_data=[]
        index= routing.Start(vehicle)
        route= manager.IndexToNode(index)
        dim= dimension.CumulVar(index)
        route_data.append([route,solution.Min(dim),solution.Max(dim)])
        while not routing.IsEnd(index):
            index= solution.Value(routing.NextVar(index))
            route= manager.IndexToNode(index)
            dim= dimension.CumulVar(index)
            route_data.append([route,solution.Min(dim),solution.Max(dim)])
        all_list.append(route_data)
    return all_list


get_routes(solution,routing,manager,time_dimension)     

### Export Results

In [ ]:
a=get_routes(solution, routing, manager, time_dimension)


import pandas as pd

a.to_excel('a.xlsx')